# AIC HCMC 2025 — PhoWhisper-large transcript

**Before running: `Kernel → Restart & Clear Outputs`**

Vietnamese transcript using [PhoWhisper-large](https://huggingface.co/vinai/PhoWhisper-large)
(Whisper-large 1.55B fine-tuned on 844h of Vietnamese; WER 4.67 on VIVOS).

- Video: `L27_V010.mp4` (Kaggle dataset `tommr211/aic-batch1-videos`)
- Output: `/kaggle/working/L27_V010_transcript.txt`

**Internet must be enabled** (model download). GPU recommended.

## 1. Setup

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("transformers", "accelerate")

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu:", torch.cuda.is_available())

## 2. Extract 16 kHz mono audio

In [ ]:
VIDEO = "/kaggle/input/datasets/tommr211/aic-batch1-videos/Videos_L27_a/video/L27_V010.mp4"
WAV = "/kaggle/working/L27_V010.wav"

subprocess.run(
    ["ffmpeg", "-y", "-i", VIDEO, "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", WAV],
    check=True, capture_output=True,
)
print("wav written:", WAV)

## 3. Transcribe with PhoWhisper-large

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="vinai/PhoWhisper-large",
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

out = pipe(
    WAV,
    chunk_length_s=30,
    generate_kwargs={"language": "vi", "task": "transcribe"},
)
transcript = out["text"].strip()
print("TRANSCRIPT:\n" + transcript)

## 4. Save transcript

In [ ]:
out_txt = "/kaggle/working/L27_V010_transcript.txt"
with open(out_txt, "w") as f:
    f.write(transcript + "\n")
print("saved:", out_txt)